# Comparison of Memory-Based and Generator-Based Data Loading
**AI Capstone Project with Deep Learning — Final submission, Question 1 (10 points)**

How to use this notebook: upload it to the Skills Network JupyterLab, then choose
**Kernel → Restart Kernel and Run All Cells**. When every cell has finished with no red errors,
choose **File → Download** and upload the downloaded file to Question 1.

In [ ]:
# ===== Step 0: make sure the needed libraries are installed (skips anything already installed) =====
import importlib.util, subprocess, sys
for pip_name, module_name in [('numpy', 'numpy'), ('matplotlib', 'matplotlib'), ('pillow', 'PIL')]:
    if importlib.util.find_spec(module_name) is None:
        print("Installing", pip_name, "...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
print("Libraries ready.")

In [ ]:
# ===== Step 1: imports + find the satellite dataset automatically =====
import os, random, time, tarfile, zipfile, urllib.request, warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# OPTIONAL: if the dataset is not found below, paste here the dataset link
# from the download cell at the top of the ORIGINAL Coursera lab, then run again.
DATA_URL = ""

def find_dataset_root():
    """Return the folder that contains class_0_non_agri and class_1_agri."""
    starts = [".", "..", os.path.expanduser("~"), "/home", "/tmp", "/resources", "/data"]
    skip = {"proc", "sys", "site-packages", "node_modules", "__pycache__"}
    seen = set()
    for start in starts:
        start = os.path.abspath(start)
        if not os.path.isdir(start) or start in seen:
            continue
        seen.add(start)
        base = start.rstrip(os.sep).count(os.sep)
        for root, dirs, files in os.walk(start):
            if "class_0_non_agri" in dirs and "class_1_agri" in dirs:
                return root
            if root.count(os.sep) - base >= 4:
                dirs[:] = []
            else:
                dirs[:] = [d for d in dirs if not d.startswith(".") and d not in skip]
    return None

dataset_path = find_dataset_root()
if dataset_path is None and DATA_URL:
    archive = os.path.basename(DATA_URL.split("?")[0]) or "dataset_archive"
    print("Downloading dataset from", DATA_URL)
    urllib.request.urlretrieve(DATA_URL, archive)
    if zipfile.is_zipfile(archive):
        zipfile.ZipFile(archive).extractall(".")
    else:
        tarfile.open(archive).extractall(".")
    dataset_path = find_dataset_root()

if dataset_path is None:
    raise FileNotFoundError(
        "Dataset not found. Open the ORIGINAL Coursera lab, run its first cells (they download "
        "the images), then run this notebook again in the same JupyterLab. Or paste the dataset "
        "link into DATA_URL above.")

dir_non_agri = os.path.join(dataset_path, "class_0_non_agri")
dir_agri     = os.path.join(dataset_path, "class_1_agri")
print("Dataset folder:", dataset_path)
print("Non-agri images:", len(os.listdir(dir_non_agri)), "| Agri images:", len(os.listdir(dir_agri)))

## Listing and sorting the image files
We use Python's `os` library to list the file names in each class folder and sort them.

In [ ]:
non_agri_images = sorted(os.listdir(dir_non_agri))
agri_images = sorted(os.listdir(dir_agri))
print("First 5 non-agri files:", non_agri_images[:5])

## Task 1: Determine the shape (dimensions) of a single image stored in the `image_data` variable

In [ ]:
image_data = np.array(Image.open(os.path.join(dir_non_agri, non_agri_images[0])))
print("Shape of a single image:", image_data.shape)   # (height, width, colour channels)
plt.imshow(image_data); plt.title(non_agri_images[0]); plt.axis("off"); plt.show()

## Memory-based (bulk) loading
Every image is opened and stored in RAM at once. Repeated access is fast, but memory grows with the dataset size.

In [ ]:
start = time.time()
all_non_agri_images = [np.array(Image.open(os.path.join(dir_non_agri, f))) for f in non_agri_images]
elapsed = time.time() - start
size_mb = sum(img.nbytes for img in all_non_agri_images) / 1024**2
print(f"Loaded {len(all_non_agri_images)} images into memory in {elapsed:.2f} s, using about {size_mb:.1f} MB of RAM")

## Generator-based (lazy) loading
Only the file paths are kept in memory; each image is read from disk when it is needed.

In [ ]:
def image_generator(folder, file_names):
    """Yield one image at a time, only when it is requested."""
    for f in file_names:
        yield np.array(Image.open(os.path.join(folder, f)))

gen = image_generator(dir_non_agri, non_agri_images)
first_image = next(gen)
print("First image from the generator:", first_image.shape)

## Task 2: Display the first four images in the `./images_dataSAT/class_0_non_agri/` directory

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(14, 4))
for ax, f in zip(axs, non_agri_images[:4]):
    ax.imshow(Image.open(os.path.join(dir_non_agri, f)))   # loaded lazily, one by one
    ax.set_title(f[:20], fontsize=8); ax.axis("off")
plt.suptitle("First four non-agricultural images"); plt.show()

## Task 3: Create a sorted list named `agri_images_paths` with the full file paths of all images in the `dir_agri` directory

In [ ]:
agri_images_paths = sorted(os.path.join(dir_agri, f) for f in os.listdir(dir_agri))
print("First 5 agricultural image paths:")
for p in agri_images_paths[:5]:
    print(p)

## Task 4: Determine the number of images of agricultural land in the `./images_dataSAT/class_1_agri/` directory

In [ ]:
num_agri_images = len(agri_images_paths)
print("Number of agricultural land images:", num_agri_images)

## Task 5: Display the first four images of agricultural land

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(14, 4))
for ax, p in zip(axs, agri_images_paths[:4]):
    ax.imshow(Image.open(p))
    ax.set_title(os.path.basename(p)[:20], fontsize=8); ax.axis("off")
plt.suptitle("First four agricultural images"); plt.show()

## Summary
- **Memory-based loading** reads the whole dataset into RAM: very fast repeated access, but memory grows linearly with the number of images.
- **Generator-based loading** keeps only file paths and reads each image on demand: tiny memory use and scales to datasets larger than RAM, at the cost of repeated disk reads.